<img src="https://raw.githubusercontent.com/aai-services/datainsight-curriculum/main/assets/brand/banner-stage-1.png" alt="Data Insight, Stage 1: Foundations" width="100%">

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aai-services/datainsight-curriculum/blob/main/stages/stage-1/1.3-cleaning-data/lesson.ipynb)

**Lesson 1.3**

# Cleaning data

Stage 1 · Foundations  |  Week 5  |  About 10 hours with practice  |  AI use: tutor only

### Learning objectives

By the end of this lesson you will be able to:

1. Find common problems in a raw table: wrong types, hidden missing values, inconsistent labels, duplicates, and impossible values
2. Fix each problem in code, and check that the fix worked
3. Record every cleaning decision so others can judge it

**Data:** `messy_countries.csv`, a copy of Our World in Data country tables for 2010, 2015 and 2020, deliberately damaged for this lesson (CC BY 4.0).

> **How to use this notebook.** Run each cell in order with Shift+Enter. Change the code and run it again: breaking things on purpose is one of the fastest ways to learn. Questions marked *Check your understanding* are for you; they are not graded.

In [ ]:
import numpy as np
import pandas as pd

DATA = "https://raw.githubusercontent.com/aai-services/datainsight-curriculum/main/datasets/"

raw = pd.read_csv(DATA + "messy_countries.csv")
raw.head()

In [ ]:
raw.info()

**Pause and look.** `gdp_per_capita` and `population` should be numbers, but pandas read them as text (`object` or `str`). That usually means something non-numeric is hiding in the column.

We never clean the raw table in place. We work on a copy, and keep a **cleaning log**: one line per decision.

In [ ]:
df = raw.copy()
log = []

## 1. Numbers stored as text

In [ ]:
df["population"].head()

In [ ]:
df["population"] = pd.to_numeric(df["population"].str.replace(",", ""), errors="coerce")
log.append("population: removed thousands separators and converted to numbers")

assert pd.api.types.is_numeric_dtype(df["population"])
df["population"].describe()

## 2. Hidden missing values

Find the text values that stop `gdp_per_capita` from being numeric: try to convert, and look at what fails.

In [ ]:
as_number = pd.to_numeric(df["gdp_per_capita"], errors="coerce")
df.loc[as_number.isna() & df["gdp_per_capita"].notna(), "gdp_per_capita"].value_counts()

Only `..` shows up. The raw file also used `n/a`, but `read_csv` already treats common codes such as `n/a`, `NA` and empty cells as missing. It does not know that `..` means missing: that is a convention of some statistics offices, which is why you always check.

In [ ]:
df["gdp_per_capita"] = pd.to_numeric(df["gdp_per_capita"].replace({"..": np.nan}))
log.append("gdp_per_capita: treated '..' as missing ('n/a' was already read as missing), converted to numbers")

assert pd.api.types.is_numeric_dtype(df["gdp_per_capita"])

Missing values can also hide as numbers. Life expectancy cannot be negative:

In [ ]:
(df["life_expectancy"] < 0).sum()

In [ ]:
df.loc[df["life_expectancy"] == -1, "life_expectancy"] = np.nan
log.append("life_expectancy: -1 is a missing-value code; set to missing")

assert (df["life_expectancy"].dropna() > 0).all()

> **Check your understanding.** Why is `errors="coerce"` useful for finding problems, and why can it be risky in your final cleaning code?

<details>
<summary>Show answer</summary>

It turns anything that cannot be converted into `NaN`, so you can list the values that failed. Used blindly, it silently hides every problem as a missing value, so always look at what was coerced.

</details>

## 3. Inconsistent labels

In [ ]:
df["region"].value_counts()

The same six regions are spelled several ways. Remove surrounding spaces, then use one consistent capitalisation.

In [ ]:
df["region"] = df["region"].str.strip().str.title()
log.append("region: stripped spaces and standardised capitalisation")

assert df["region"].nunique() == 6, df["region"].unique()
df["region"].value_counts()

## 4. Duplicates

Exact duplicate rows would count a country twice in every total and average.

In [ ]:
print("exact duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates()
log.append("removed exact duplicate rows")

# A stronger check: each country should appear once per year
assert not df.duplicated(subset=["code", "year"]).any()
len(df)

## 5. Impossible values

Look at the extremes. The highest life expectancy recorded for any country is in the mid 80s.

In [ ]:
df.sort_values("life_expectancy", ascending=False).head(3)[["country", "year", "life_expectancy"]]

One value is roughly ten times too large, most likely a typing slip. We have two honest options: set it to missing, or correct it and say so. Correcting it assumes we know what happened, so the safer choice is to set it to missing and record why.

In [ ]:
impossible = df["life_expectancy"] > 100
print(df.loc[impossible, ["country", "year", "life_expectancy"]])
df.loc[impossible, "life_expectancy"] = np.nan
log.append("life_expectancy: one value above 100 (Kenya, 2015) is impossible; set to missing")

assert df["life_expectancy"].max() < 100

> **Check your understanding.** Why set the impossible life expectancy to missing instead of dividing it by ten?

<details>
<summary>Show answer</summary>

Dividing by ten assumes we know exactly what went wrong. Setting it to missing makes no such assumption, and the cleaning log tells readers what was done and why.

</details>

## 6. The cleaning log

This is part of your results, not an afterthought. A reader should be able to reproduce your table and disagree with a decision if they have reason to.

In [ ]:
for i, step in enumerate(log, start=1):
    print(f"{i}. {step}")

print(f"\nRows: {len(raw)} raw, {len(df)} clean")

## 7. Worked example: did cleaning change the answer?

Compare average life expectancy by region in 2020 before and after cleaning. The raw table cannot even be grouped correctly, because of the inconsistent region names and the `-1` codes.

In [ ]:
before = raw[raw["year"] == 2020].groupby("region")["life_expectancy"].mean().round(1)
after = df[df["year"] == 2020].groupby("region")["life_expectancy"].mean().round(1)
print("Before cleaning:\n", before, "\n")
print("After cleaning:\n", after)

Before cleaning there are more than six "regions", and the `-1` codes drag some averages down. Unchecked, those errors would have gone straight into a chart.

## Key takeaways

- Never clean the raw table in place: work on a copy and keep the original.
- Missing values hide in many forms: `..`, `-1`, empty text, and more.
- Every cleaning step should end with a check that would fail if the step went wrong.
- The cleaning log is part of your results, not an afterthought.

## Next step

Open `practice-1.3.ipynb`, which uses a different damaged table.

---

<img src="https://raw.githubusercontent.com/aai-services/datainsight-curriculum/main/assets/brand/mark.png" alt="Data Insight" width="32">

**Data Insight** · Open data science curriculum · [datainsightonline.com](https://datainsightonline.com)

Released under the Creative Commons Attribution 4.0 license (CC BY 4.0).

Suggested citation: Data Insight (2026). *Lesson 1.3: Cleaning data*. https://github.com/aai-services/datainsight-curriculum